# May 25 — TFIM-QRC Light-Touch Prototype

This notebook runs the smallest end-to-end exact-state TFIM quantum reservoir prototype for the Phase 2 volatility-forecasting project.

Scope:

- target: `future_rv_20d`;
- fallback architecture: 6 qubits, PCA-6, 6 temporal anchors, Z-only exact expectations;
- readout: Ridge regression on log-volatility;
- metrics: RMSE, QLIKE, Mincer-Zarnowitz.

This is viability evidence and architecture validation, not a final performance benchmark.

In [2]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Load data and build PCA-6 sequence windows

In [3]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

pca6.explained_variance

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308
5,6,0.055584,0.804892


In [4]:
{name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits.items()}

{'train': ((5420, 40, 6), (5420,)),
 'val': ((1219, 40, 6), (1219,)),
 'test': ((1019, 40, 6), (1019,))}

## 2. Run fallback 6-qubit QRC prototype

In [5]:
fallback_config = TFIMQRCConfig(
    qubits=6,
    pca_components=6,
    lookback_days=40,
    anchor_count=6,
    anchor_policy="even",
    observable_mode="z",
    trotter_steps_per_anchor=1,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    ridge_alpha=10.0,
    target_transform="log",
    seed=42,
)

fallback_result = fit_tfim_qrc_regressor(
    sequence_splits,
    config=fallback_config,
    target=target,
    verbose=True,
)

fallback_summary = pd.DataFrame([summarize_qrc_result(fallback_result)])
fallback_summary.T

QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019


,0
qubits,6
pca_components,6
lookback_days,40
anchor_count,6
anchor_policy,even
observable_mode,z
trotter_steps_per_anchor,1
coupling_scale,0.7
transverse_field,0.5
evolution_time,0.5


## 3. Optional small observable probe

In [6]:
probe_rows = []

for observable_mode in ["z", "zx", "zxzz"]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode=observable_mode,
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=10.0,
        target_transform="log",
        seed=42,
    )
    print(f"Running observable_mode={observable_mode}")
    result = fit_tfim_qrc_regressor(
        sequence_splits,
        config=config,
        target=target,
        verbose=False,
    )
    probe_rows.append(summarize_qrc_result(result))

observable_probe = pd.DataFrame(probe_rows).sort_values("val_rmse")
observable_probe[[
    "observable_mode", "n_reservoir_features",
    "train_rmse", "val_rmse", "test_rmse",
    "train_qlike", "val_qlike", "test_qlike",
    "train_mz_r2", "val_mz_r2", "test_mz_r2",
]]

Running observable_mode=z
Running observable_mode=zx
Running observable_mode=zxzz


,observable_mode,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,z,6,0.097432,0.067361,0.107953,-2.178178,-2.979082,-1.863498,0.069312,0.006497,0.013055
2,zxzz,17,0.094534,0.069317,0.105705,-2.270258,-2.961464,-1.961164,0.124484,0.017802,0.038175
1,zx,12,0.095233,0.069470,0.105713,-2.247461,-2.963106,-1.963174,0.111488,0.008869,0.037934


## 4. Save results

In [7]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

fallback_summary.to_csv(out_dir / "phase2_tfim_qrc_fallback_summary.csv", index=False)
observable_probe.to_csv(out_dir / "phase2_tfim_qrc_observable_probe.csv", index=False)
pca6.explained_variance.to_csv(out_dir / "phase2_qrc_pca6_explained_variance.csv", index=False)

print("Saved QRC prototype outputs to", out_dir)

Saved QRC prototype outputs to results/tables


## 5. Interpretation template

Use after running the notebook:

```text
The fallback TFIM-QRC prototype runs end-to-end with train-only PCA inputs, compressed 40-day temporal memory, exact observable expectations, and a ridge readout on log realized volatility. This validates the architecture interface: data preprocessing, temporal encoding, quantum reservoir feature extraction, classical readout, and Track A metric evaluation. The result should be compared against persistence, HAR/Ridge/ElasticNet, and the PCA-compressed ESN reservoir baseline. Performance is secondary at this milestone; the main purpose is to establish a reproducible QRC implementation path and identify the most useful next design probe.
```

In [8]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca8 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=8,
    prefix="pca8",
)

sequence_splits_8 = make_qrc_sequence_splits(
    pca8.splits,
    feature_columns=pca8.feature_columns,
    target_column=target,
    lookback_days=40,
)

pca8.explained_variance

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308
5,6,0.055584,0.804892
6,7,0.036496,0.841388
7,8,0.032709,0.874098


In [9]:
{name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_8.items()}

{'train': ((5420, 40, 8), (5420,)),
 'val': ((1219, 40, 8), (1219,)),
 'test': ((1019, 40, 8), (1019,))}

In [10]:
qrc8_config = TFIMQRCConfig(
    qubits=8,
    pca_components=8,
    lookback_days=40,
    anchor_count=8,
    anchor_policy="even",
    observable_mode="zxzz",
    trotter_steps_per_anchor=1,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    ridge_alpha=10.0,
    target_transform="log",
    seed=42,
)

qrc8_result = fit_tfim_qrc_regressor(
    sequence_splits_8,
    config=qrc8_config,
    target=target,
    verbose=True,
)

qrc8_summary = pd.DataFrame([summarize_qrc_result(qrc8_result)])
qrc8_summary.T

QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019


,0
qubits,8
pca_components,8
lookback_days,40
anchor_count,8
anchor_policy,even
observable_mode,zxzz
trotter_steps_per_anchor,1
coupling_scale,0.7
transverse_field,0.5
evolution_time,0.5


In [11]:
# Optional: compare observables at PCA-8 / 8 qubits / 8 anchors

probe_rows_8 = []

for observable_mode in ["z", "zx", "zxzz"]:
    config = TFIMQRCConfig(
        qubits=8,
        pca_components=8,
        lookback_days=40,
        anchor_count=8,
        anchor_policy="even",
        observable_mode=observable_mode,
        trotter_steps_per_anchor=1,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        ridge_alpha=10.0,
        target_transform="log",
        seed=42,
    )

    print(f"Running PCA-8 / 8q / 8 anchors / observable_mode={observable_mode}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_8,
        config=config,
        target=target,
        verbose=True,
    )

    probe_rows_8.append(summarize_qrc_result(result))

observable_probe_8 = pd.DataFrame(probe_rows_8).sort_values("val_rmse")

observable_probe_8[
    [
        "observable_mode",
        "n_reservoir_features",
        "train_rmse",
        "val_rmse",
        "test_rmse",
        "train_qlike",
        "val_qlike",
        "test_qlike",
        "train_mz_r2",
        "val_mz_r2",
        "test_mz_r2",
    ]
]

Running PCA-8 / 8q / 8 anchors / observable_mode=z
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running PCA-8 / 8q / 8 anchors / observable_mode=zx
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/54

,observable_mode,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,z,8,0.100114,0.065724,0.109380,-2.135112,-3.006526,-1.831909,0.022414,0.010244,0.004998
2,zxzz,23,0.097706,0.068201,0.109260,-2.194921,-2.983476,-1.852447,0.068716,0.006665,0.008340
1,zx,16,0.098450,0.068205,0.109964,-2.173719,-2.982605,-1.826196,0.053633,0.004870,0.001034


In [12]:
# Save outputs

out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

qrc8_summary.to_csv(
    out_dir / "phase2_tfim_qrc_pca8_8q_8anchor_zxzz_summary.csv",
    index=False,
)

observable_probe_8.to_csv(
    out_dir / "phase2_tfim_qrc_pca8_8q_8anchor_observable_probe.csv",
    index=False,
)

pca8.explained_variance.to_csv(
    out_dir / "phase2_qrc_pca8_explained_variance.csv",
    index=False,
)

print("Saved PCA-8 / 8q QRC outputs to", out_dir)

Saved PCA-8 / 8q QRC outputs to results/tables
